# Chapter 26 — MongoDB concepts: documents, collections, schema design, and aggregation

**Book role:** Databases  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Explain documents, collections, BSON, and flexible schema.
- Choose embedding or referencing based on access patterns and update boundaries.
- Write filters, projections, sorting, and aggregation pipelines.
- Normalize results for data-frame analysis.

## 1. Documents and flexible schema are not schema-free

MongoDB stores BSON documents in collections. Different documents can have different fields, which supports evolving models, but applications still need a contract for required fields, allowed types, and naming. A flexible schema does not remove the need for data validation; it can simply move more of that responsibility into application code or collection validators.

Design around query patterns. Embed a small bounded set of child information when it is usually read and updated with the parent; reference separate documents when data is shared, large, independently updated, or unbounded. Avoid embedding ever-growing arrays that can make documents expensive to update or exceed storage limits. Document the grain and relationships even when there are no foreign-key constraints equivalent to a relational database.

### Worked example 1: Create a simple aggregation pipeline

This pipeline is illustrative: real BSON dates should normally be stored as dates rather than ISO strings, and the `$addToSet` array can consume memory. A simpler `$sum: 1` works if documents represent one order each; choose based on the collection grain.

In [ ]:
pipeline = [
    {"$match": {"status": "Paid", "order_date": {"$gte": "2026-01-01", "$lt": "2027-01-01"}}},
    {"$group": {
        "_id": "$region",
        "order_count": {"$addToSet": "$order_id"},
        "revenue": {"$sum": {"$multiply": ["$quantity", "$unit_price"]}},
    }},
    {"$project": {
        "_id": 0,
        "region": "$_id",
        "order_count": {"$size": "$order_count"},
        "revenue": 1,
    }},
    {"$sort": {"revenue": -1}},
]
print(pipeline)
# Execute with collection.aggregate(pipeline) after aligning field names and date types with your schema.

## 2. Filters, projections, and indexes

A `find()` filter selects matching documents; a projection controls returned fields; sort and limit shape results. Query documents should come from validated UI criteria, not be built from arbitrary raw strings. MongoDB operators such as `$gte`, `$in`, `$and`, and `$or` express filter semantics. Watch for the difference between a missing field and a field explicitly set to `null` when data quality matters.

Indexes can make common queries faster but increase storage and write overhead. Index fields used frequently in filters and sorts, then inspect query plans and production-like data. Avoid creating an index for every possible GUI filter combination. A filterable dashboard should start with a small number of known access paths and be measured before adding indexes.

### Worked example 2: Map UI criteria to a safe Mongo filter

Use BSON datetime objects when the collection stores BSON dates. The GUI should parse and validate user dates first. For complex criteria, construct a small known set of operators instead of accepting an arbitrary query document from user text.

In [ ]:
def build_mongo_filter(region: str, start_date, end_date, include_cancelled: bool) -> dict:
    criteria = {"order_date": {"$gte": start_date, "$lt": end_date}}
    if region != "All":
        criteria["region"] = region
    if not include_cancelled:
        criteria["status"] = {"$ne": "Cancelled"}
    return criteria

print(build_mongo_filter("North", "2026-01-01", "2027-01-01", False))

## 3. Aggregation pipelines and the DataFrame boundary

An aggregation pipeline applies stages such as `$match`, `$project`, `$unwind`, `$group`, `$sort`, and `$limit`. Put restrictive `$match` stages early when semantics allow, and use `$project` to omit fields that the GUI does not need. `$unwind` turns array elements into separate pipeline documents; it changes row grain and can multiply rows just as a relational join can.

The result may contain nested documents, arrays, ObjectIds, dates, and Decimal128 values. Normalize these into stable columns before constructing a pandas DataFrame. Preserve an identifier as a string for display or as an ObjectId in the repository layer; be explicit about decimal conversion to avoid silently losing financial precision.

## Guided lab: Lab: Mongo analytics pipeline

Design documents for customers, orders, and order lines. State what is embedded and referenced and why. Implement filters for date, region, and status; an aggregation for sales/profit by region; and a detail-query projection. Convert aggregation output into a fixed schema and compare the grouped totals to the pandas implementation.

## Review questions and practice

1. Why is flexible schema not the same as no schema?
2. How can `$unwind` change the row grain?
3. Why can embedding an unbounded array be a bad idea?
4. When does an index help and what does it cost?
5. Why should BSON date and decimal types be normalized carefully?

## Further reading

- MongoDB data modeling: https://www.mongodb.com/docs/manual/data-modeling/
- MongoDB aggregation: https://www.mongodb.com/docs/manual/aggregation/
- MongoDB indexes: https://www.mongodb.com/docs/manual/indexes/

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.